# 🌫️ Air Quality Prediction using TensorFlow ANN

## Project Overview

This project uses **one Air Quality dataset** to implement both:

1. 🧠 **ANN Classification** → Predict `AQI_Bucket`
2. 📈 **ANN Regression** → Predict numerical `AQI`

### Technologies
- Python
- Pandas
- NumPy
- Matplotlib
- Scikit-learn
- TensorFlow / Keras
- Pickle

The same 8 input features are used for both models:
`PM2.5`, `PM10`, `Ozone`, `NO2`, `CO`, `SO2`, `Temperature`, `Humidity`.


## 1. Import Required Libraries

We import libraries for data analysis, preprocessing, model building, and evaluation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout

import pickle

print("TensorFlow version:", tf.__version__)


## 2. Load the Dataset

The dataset contains date, city, AQI, pollutant values, temperature, and humidity.


In [ ]:
df = pd.read_csv("air_quality_dataset.csv")

df.head()


## 3. Understand the Dataset

Check the column names, number of rows and columns, and data types.


In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nShape:", df.shape)

print("\nDataset Information:")
df.info()


## 4. Check Missing Values

Missing values can cause problems during scaling and model training, so we check them before preprocessing.


In [ ]:
df.isnull().sum()


## 5. Remove Missing Values

For this project, rows containing missing values are removed.


In [ ]:
df = df.dropna().copy()

print("Shape after removing missing values:", df.shape)


## 6. Check and Remove Duplicate Rows

Duplicate rows can affect model training, so we identify and remove them.


In [ ]:
print("Duplicate rows:", df.duplicated().sum())

df = df.drop_duplicates().copy()

print("Shape after removing duplicates:", df.shape)


## 7. Exploratory Data Analysis

### AQI Distribution

This plot shows how AQI values are distributed in the dataset.


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["AQI"], bins=30)
plt.xlabel("AQI")
plt.ylabel("Frequency")
plt.title("AQI Distribution")
plt.show()


## 8. Create Classification Target

The numerical `AQI` value is converted into air-quality categories.

- 0–49 → Good
- 50–99 → Moderate
- 100–149 → Unhealthy for Sensitive Groups
- 150–199 → Unhealthy
- 200+ → Very Unhealthy

This `AQI_Bucket` becomes the target for ANN Classification.


In [ ]:
df["AQI_Bucket"] = pd.cut(
    df["AQI"],
    bins=[0, 50, 100, 150, 200, float("inf")],
    labels=[
        "Good",
        "Moderate",
        "Unhealthy for Sensitive Groups",
        "Unhealthy",
        "Very Unhealthy"
    ],
    right=False
)

print(df["AQI_Bucket"].value_counts())


# 🧠 PART A — ANN CLASSIFICATION

## 9. Select Classification Features and Target

We exclude `Date`, `City`, `AQI`, and `AQI_Bucket` from the input features.

The model uses exactly these **8 features**:
1. PM2.5
2. PM10
3. Ozone
4. NO2
5. CO
6. SO2
7. Temperature
8. Humidity


In [ ]:
classification_features = [
    "PM2.5",
    "PM10",
    "Ozone",
    "NO2",
    "CO",
    "SO2",
    "Temperature",
    "Humidity"
]

X_class = df[classification_features]
y_class = df["AQI_Bucket"]

print("Classification features:")
print(X_class.columns.tolist())
print("Number of features:", X_class.shape[1])


## 10. Encode the Classification Target

Neural networks work with numerical targets, so the category names are converted into numbers using `LabelEncoder`.


In [ ]:
encoder = LabelEncoder()

y_class_encoded = encoder.fit_transform(y_class)

print("Encoded classes:")
print(encoder.classes_)


## 11. Split Classification Data into Training and Testing Sets

80% of the data is used for training and 20% for testing.


In [ ]:
X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(
    X_class,
    y_class_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_class_encoded
)

print("Training shape:", X_train_class.shape)
print("Testing shape:", X_test_class.shape)


## 12. Scale Classification Features

`StandardScaler` puts the numerical features on a comparable scale, which helps ANN training.


In [ ]:
classification_scaler = StandardScaler()

X_train_class = classification_scaler.fit_transform(X_train_class)
X_test_class = classification_scaler.transform(X_test_class)


## 13. Build the ANN Classification Model

The network contains:
- Input layer with 8 features
- Dense hidden layer with 64 neurons
- Dense hidden layer with 32 neurons
- Dropout layer
- Softmax output layer for multiple classes


In [ ]:
classification_model = Sequential([
    Input(shape=(X_train_class.shape[1],)),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(len(np.unique(y_class_encoded)), activation="softmax")
])

classification_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

classification_model.summary()


## 14. Train the ANN Classification Model

The model is trained for 50 epochs with a batch size of 32.


In [ ]:
history_class = classification_model.fit(
    X_train_class,
    y_train_class,
    epochs=50,
    batch_size=32,
    validation_split=0.2
)


## 15. Plot Classification Training Performance

These plots help us observe training and validation accuracy/loss.


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_class.history["accuracy"], label="Training Accuracy")
plt.plot(history_class.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("ANN Classification Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_class.history["loss"], label="Training Loss")
plt.plot(history_class.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ANN Classification Loss")
plt.legend()
plt.show()


## 16. Evaluate Classification Model

Accuracy measures the percentage of correctly classified test samples.


In [ ]:
class_loss, class_accuracy = classification_model.evaluate(
    X_test_class,
    y_test_class,
    verbose=0
)

print("Classification Test Accuracy:", class_accuracy)


## 17. Classification Report and Confusion Matrix


In [ ]:
class_probabilities = classification_model.predict(
    X_test_class,
    verbose=0
)

y_pred_class = np.argmax(class_probabilities, axis=1)

print(classification_report(
    y_test_class,
    y_pred_class,
    labels=np.unique(y_test_class),
    target_names=encoder.inverse_transform(np.unique(y_test_class)),
    zero_division=0
))

cm = confusion_matrix(y_test_class, y_pred_class)

plt.figure(figsize=(7, 5))
plt.imshow(cm)
plt.title("Classification Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.colorbar()
plt.show()


## 18. Save Classification Model, Scaler, and Encoder

These files will later be used by the Streamlit application.


In [ ]:
classification_model.save("classification_model.h5")

with open("classification_scaler.pkl", "wb") as file:
    pickle.dump(classification_scaler, file)

with open("classification_encoder.pkl", "wb") as file:
    pickle.dump(encoder, file)

print("Classification model and preprocessing files saved.")


# 📈 PART B — ANN REGRESSION

## 19. Select Regression Features and Target

The same 8 input features are used, but now the target is the numerical `AQI`.


In [ ]:
regression_features = [
    "PM2.5",
    "PM10",
    "Ozone",
    "NO2",
    "CO",
    "SO2",
    "Temperature",
    "Humidity"
]

X_reg = df[regression_features]
y_reg = df["AQI"]

print("Regression features:")
print(X_reg.columns.tolist())
print("Number of features:", X_reg.shape[1])


## 20. Split Regression Data into Training and Testing Sets


In [ ]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

print("Training shape:", X_train_reg.shape)
print("Testing shape:", X_test_reg.shape)


## 21. Scale Regression Features


In [ ]:
regression_scaler = StandardScaler()

X_train_reg = regression_scaler.fit_transform(X_train_reg)
X_test_reg = regression_scaler.transform(X_test_reg)


## 22. Build the ANN Regression Model

The output layer contains one neuron because the model predicts one continuous value: `AQI`.


In [ ]:
regression_model = Sequential([
    Input(shape=(X_train_reg.shape[1],)),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="linear")
])

regression_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

regression_model.summary()


## 23. Train the ANN Regression Model


In [ ]:
history_reg = regression_model.fit(
    X_train_reg,
    y_train_reg,
    epochs=50,
    batch_size=32,
    validation_split=0.2
)


## 24. Plot Regression Training Performance


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_reg.history["loss"], label="Training Loss")
plt.plot(history_reg.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("ANN Regression Loss")
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_reg.history["mae"], label="Training MAE")
plt.plot(history_reg.history["val_mae"], label="Validation MAE")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.title("ANN Regression MAE")
plt.legend()
plt.show()


## 25. Predict AQI with the Regression Model


In [ ]:
y_pred_reg = regression_model.predict(
    X_test_reg,
    verbose=0
).ravel()

print("First 10 predictions:")
print(y_pred_reg[:10])


## 26. Evaluate Regression Model

Regression evaluation uses:
- **MAE** — Mean Absolute Error
- **MSE** — Mean Squared Error
- **RMSE** — Root Mean Squared Error
- **R²** — R-squared score


In [ ]:
mae = mean_absolute_error(y_test_reg, y_pred_reg)
mse = mean_squared_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_reg, y_pred_reg)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)


## 27. Actual vs Predicted AQI

This scatter plot compares the real AQI values with ANN predictions.


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_test_reg, y_pred_reg)
plt.xlabel("Actual AQI")
plt.ylabel("Predicted AQI")
plt.title("Actual vs Predicted AQI")
plt.show()


## 28. Save Regression Model and Scaler

These files will be used by Streamlit for AQI prediction.


In [ ]:
regression_model.save("regression_model.h5")

with open("regression_scaler.pkl", "wb") as file:
    pickle.dump(regression_scaler, file)

print("Regression model and scaler saved.")


# 💾 Final Project Files

After running the notebook, these files should be available:

```text
classification_model.h5
classification_scaler.pkl
classification_encoder.pkl

regression_model.h5
regression_scaler.pkl
```

The same 8 features are used in both models:

```text
PM2.5
PM10
Ozone
NO2
CO
SO2
Temperature
Humidity
```


# 🌐 Streamlit Deployment Preparation

Use `compile=False` while loading the `.h5` models in Streamlit. This avoids Keras legacy HDF5 loss/metric deserialization issues during prediction.

The Streamlit input order must exactly match the 8 training features:

`PM2.5 → PM10 → Ozone → NO2 → CO → SO2 → Temperature → Humidity`


## `requirements.txt`

```text
streamlit
tensorflow
pandas
scikit-learn
numpy
scipy
h5py
protobuf
```
